# Chapter 7 - Lab 3: <font color='blue'>Conversational Multi-Agent Analysis with AutoGen</font>

**<font color='purple'>Goal</font>**:
Solve a financial-analysis task with a third coordination philosophy: **conversation**. An `AssistantAgent` writes Python code using `yfinance`; a `UserProxyAgent` executes that code and feeds the results back. There are no explicit handoffs (`can_handoff_to`) and no shared state object (`Context`) — the conversation history *is* the coordination mechanism.

This is AutoGen's native strength: open-ended tasks where an agent writes and runs its own analysis code.

**<font color='purple'>Tech stack</font>**:

* **AutoGen classic 0.2 API** (`pyautogen<0.4`) — `AssistantAgent`, `UserProxyAgent`, `initiate_chat`.
* **`LocalCommandLineCodeExecutor`** — runs the generated code in a working directory.
* **OpenAI** `gpt-4o`.
* **`yfinance`** — live market data fetched by the *generated* code (no API key needed).

A version note: this lab uses the classic AutoGen 0.2-style API, which remains the most widely deployed and most documented lineage. AutoGen 0.4 (January 2025) was a ground-up rewrite with an async, event-driven API, and Microsoft has since folded the project's direction into the Microsoft Agent Framework; the community fork AG2 continues the 0.2 lineage. Chapter 3 covers this landscape and the migration paths — the coordination concepts shown here carry over to all of them.

## 1. Install packages

In [ ]:
%pip install -q "pyautogen<0.4" yfinance matplotlib python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally — assume the env var is already set.
    pass

## 3. Configure the model and the code executor

Two configuration objects: the LLM config (model plus `cache_seed`, which enables AutoGen's response cache so repeated runs are cheap and reproducible) and a `LocalCommandLineCodeExecutor` that runs generated code inside a scratch working directory. Everything the generated code saves — for example, plot images — lands in that directory.

One portability detail: the executor runs code with whatever `python` is first on the `PATH`, which is not necessarily the interpreter this notebook (and its `%pip` installs) are using. Prepending this kernel's interpreter directory to the `PATH` makes the generated code run in the same environment — on Colab and locally.

In [ ]:
import sys
import tempfile
from autogen.coding import LocalCommandLineCodeExecutor

# Make the executor's `python` this kernel's interpreter.
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ['PATH']

config_list = [{'model': 'gpt-4o', 'api_key': os.environ['OPENAI_API_KEY']}]
llm_config = {'config_list': config_list, 'cache_seed': 42}

work_dir = tempfile.mkdtemp()
executor = LocalCommandLineCodeExecutor(timeout=60, work_dir=work_dir)
print(f'Code executor working directory: {work_dir}')

## 4. Define the two agents

The `AssistantAgent` is LLM-powered: it plans the analysis and writes the code. The `UserProxyAgent` is not — it executes whatever code blocks arrive and returns the output. Two details in the system message keep the conversation well-behaved: plots must be **saved to files** (the executor has no display), and `TERMINATE` may only be written **after** the assistant has seen execution results — otherwise the termination check would end the chat before any code runs.

In [ ]:
from autogen import AssistantAgent, UserProxyAgent

assistant = AssistantAgent(
    name='assistant',
    llm_config=llm_config,
    system_message=(
        'You are a financial analyst assistant. When asked to analyze '
        'a company, write Python code using yfinance to fetch financial '
        'data and compute key metrics. Present results in a clear, '
        'structured format. Save any plots to PNG files instead of '
        'calling plt.show(). After you have seen the execution results '
        'and written your final analysis, end that final message with '
        'TERMINATE. Never write TERMINATE in a message that contains code.'
    ),
)

code_executor = UserProxyAgent(
    name='code_executor',
    human_input_mode='NEVER',
    code_execution_config={'executor': executor},
    llm_config=False,
    is_termination_msg=lambda msg: 'TERMINATE' in (msg.get('content') or ''),
)

## 5. Run the conversation

`initiate_chat` starts the loop: the proxy sends the task, the assistant replies with code, the proxy executes it and returns the output, and the assistant either fixes errors or writes the final analysis. `max_turns=5` is a hard stop in case the model never says `TERMINATE`.

In [ ]:
query = ('What is the current price and key financial metrics of NVIDIA? '
         'Plot the stock price over the last 6 months.')

chat_result = code_executor.initiate_chat(
    assistant,
    message=query,
    max_turns=5,
)

## 6. Inspect the final answer, the cost, and any plots

AutoGen tracks spend per conversation through `chat_result.cost` — a dictionary of total cost and per-model token counts. This per-conversation transparency is valuable in multi-agent systems, where token costs multiply quickly. The final analysis is in `chat_result.summary`, and any PNG the generated code saved is in the executor's working directory.

In [ ]:
print('FINAL ANALYSIS:')
print(chat_result.summary.replace('TERMINATE', '').strip())
print()
print('COST:')
print(chat_result.cost)

In [ ]:
import glob
from IPython.display import Image, display

for png in sorted(glob.glob(os.path.join(work_dir, '*.png'))):
    print(png)
    display(Image(filename=png))

## 7. Results

Same domain as Labs 1 and 2; a completely different coordination model. **What to notice about conversational multi-agent systems:**

* **Conversation is the coordination mechanism.** No handoff declarations, no shared state object — agents influence each other by reading and responding to the message history.
* **Code execution closes the loop.** The assistant does not just describe an analysis; it writes code, sees real output (or a real traceback), and iterates. Error recovery comes free: a failed run goes back to the assistant as text.
* **Termination needs care.** The chat ends when the assistant says `TERMINATE` — which is why the system message forbids `TERMINATE` in the same message as code, and why `max_turns` provides a hard stop. Unbounded conversations are the conversational analogue of Lab 1's iteration-budget problem.
* **Flexible but less predictable.** This model shines for open-ended tasks (research, exploratory analysis) and is harder to audit for structured workflows — the trade-off the chapter's framework comparison table summarizes.